In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print("====== PREPARING NATURAL-CORRUPTION BENCHMARK AND APRIL-GAN ======")
BENCHMARK_REPOSITORY = "Parsagh05/Natural-Corruption-Robustness"
BENCHMARK_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
APRILGAN_ROOT = Path("/kaggle/working/VAND-APRIL-GAN")
APRILGAN_COMMIT = "f13b8a634e04f9fde8fa03db125b25af5695d8e1"

if not BENCHMARK_ROOT.exists():
    subprocess.run(
        ["git", "clone", f"https://github.com/{BENCHMARK_REPOSITORY}.git", str(BENCHMARK_ROOT)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(BENCHMARK_ROOT), "pull", "--ff-only"], check=True
    )

required_wrapper = BENCHMARK_ROOT / "zero_shot/harness/models.py"
if not required_wrapper.is_file() or "APRILGANWrapper" not in required_wrapper.read_text(encoding="utf-8"):
    raise RuntimeError(
        "The cloned benchmark revision does not yet contain APRIL-GAN support. "
        "Commit and push these local changes before running on Kaggle."
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "-r",
        str(BENCHMARK_ROOT / "zero_shot/requirements.txt"),
    ],
    check=True,
)

if not APRILGAN_ROOT.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ByChelsea/VAND-APRIL-GAN.git", str(APRILGAN_ROOT)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(APRILGAN_ROOT), "fetch", "origin"], check=True
    )
subprocess.run(
    ["git", "-C", str(APRILGAN_ROOT), "checkout", "--detach", APRILGAN_COMMIT],
    check=True,
)
resolved_commit = subprocess.run(
    ["git", "-C", str(APRILGAN_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if resolved_commit != APRILGAN_COMMIT:
    raise RuntimeError(f"Wrong APRIL-GAN source commit: {resolved_commit}")

required_upstream = [
    APRILGAN_ROOT / "open_clip" / "factory.py",
    APRILGAN_ROOT / "model.py",
    APRILGAN_ROOT / "prompt_ensemble.py",
    APRILGAN_ROOT / "exps" / "pretrained" / "mvtec_pretrained.pth",
    APRILGAN_ROOT / "exps" / "pretrained" / "visa_pretrained.pth",
]
missing_upstream = [str(path) for path in required_upstream if not path.is_file()]
if missing_upstream:
    raise FileNotFoundError(
        "APRIL-GAN clone/checkpoints are incomplete:\n  - " + "\n  - ".join(missing_upstream)
    )

for import_path in (BENCHMARK_ROOT, APRILGAN_ROOT):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
os.environ["APRILGAN_ROOT"] = str(APRILGAN_ROOT)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running APRIL-GAN.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Official APRIL-GAN commit: {resolved_commit}")
print("Released cross-dataset checkpoints found in exps/pretrained.")


# APRIL-GAN zero-shot natural-corruption benchmark

This notebook pins the official [ByChelsea/VAND-APRIL-GAN](https://github.com/ByChelsea/VAND-APRIL-GAN) implementation and uses the checkpoints committed in `exps/pretrained`. It follows the released cross-dataset protocol: VisA-trained projections evaluate MVTec AD and MVTec-trained projections evaluate VisA. The OpenAI ViT-L/14@336px backbone is downloaded once unless attached as a Kaggle input.


In [ ]:
import gc

from shared import corruption_plan_path
from zero_shot.harness.runner import run_evaluation

# Choose exactly one target. APRIL-GAN automatically enforces the official
# opposite-dataset checkpoint: VisA weights -> MVTec, MVTec weights -> VisA.
DATASET_NAME = "visa"  # "mvtec" or "visa"
DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa"}:
    raise ValueError("DATASET_NAME must be 'mvtec' or 'visa'.")
IS_MVTEC = DATASET_NAME == "mvtec"
WEIGHT_DATASET = "visa" if IS_MVTEC else "mvtec"

MVTEC_ROOT = "/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection"
VISA_ROOT = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"
OUTPUT_ROOT = "/kaggle/working/outputs"
CLIP_DOWNLOAD_DIR = "/kaggle/working/aprilgan-clip"

# Optional offline backbone. If it is not attached, Internet must be enabled
# and the official OpenAI ViT-L/14@336px file is downloaded once.
clip_candidates = []
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    clip_candidates.extend(kaggle_input.rglob("ViT-L-14-336px.pt"))
CLIP_WEIGHT_PATH = str(next((path for path in clip_candidates if path.is_file()), ""))

USE_CATEGORIZED_CORRUPTIONS = True
CORRUPTION_SEED = 123
UNCATEGORIZED_CORRUPTION_TYPES = [
    "gaussian_noise", "shot_noise", "impulse_noise", "defocus_blur",
    "motion_blur", "zoom_blur", "brightness", "contrast",
]
CATEGORIZED_CORRUPTION_TYPES = ["noise", "blur", "photometric", "geometric"]
CORRUPTION_TYPES = (
    CATEGORIZED_CORRUPTION_TYPES
    if USE_CATEGORIZED_CORRUPTIONS
    else UNCATEGORIZED_CORRUPTION_TYPES
)
INCLUDE_CLEAN_BASELINE = True
SEVERITY_LEVELS = [1, 2, 3, 4]
BATCH_SIZE = 1
CORRUPTION_CACHE_ROOT = None
CORRUPTION_CACHE_FORMAT = "png"
DEVICE = "cuda"
CORRUPTION_PLAN = corruption_plan_path(DATASET_NAME)

checkpoint_paths = {
    "mvtec": str(APRILGAN_ROOT / "exps" / "pretrained" / "mvtec_pretrained.pth"),
    "visa": str(APRILGAN_ROOT / "exps" / "pretrained" / "visa_pretrained.pth"),
}
print("LAUNCHING APRIL-GAN ZERO-SHOT ROBUSTNESS BENCHMARK")
print(f"Target / weights: {DATASET_NAME} / {WEIGHT_DATASET} (cross-dataset)")
print(f"CLIP backbone: {CLIP_WEIGHT_PATH or 'official download'}")
print(f"Corruptions: {CORRUPTION_TYPES} @ {SEVERITY_LEVELS}; clean={INCLUDE_CLEAN_BASELINE}")
print(f"Outputs: {OUTPUT_ROOT}")

run_evaluation(
    mvtec_root=MVTEC_ROOT if IS_MVTEC else None,
    visa_root=None if IS_MVTEC else VISA_ROOT,
    output_root=OUTPUT_ROOT,
    models=["APRIL-GAN"],
    model_kwargs={
        "APRIL-GAN": {
            "aprilgan_root": str(APRILGAN_ROOT),
            "checkpoint_paths": checkpoint_paths,
            "weight_dataset": WEIGHT_DATASET,
            "clip_weight_path": CLIP_WEIGHT_PATH,
            "clip_download_dir": CLIP_DOWNLOAD_DIR,
            "strict_source_commit": True,
        }
    },
    device=DEVICE,
    dataset=DATASET_NAME,
    corruption_types=CORRUPTION_TYPES,
    severity_levels=SEVERITY_LEVELS,
    include_clean=INCLUDE_CLEAN_BASELINE,
    batch_size=BATCH_SIZE,
    corruption_cache_root=CORRUPTION_CACHE_ROOT,
    corruption_cache_format=CORRUPTION_CACHE_FORMAT,
    categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
    categorized_corruption_plans={DATASET_NAME: str(CORRUPTION_PLAN)},
    corruption_seed=CORRUPTION_SEED if USE_CATEGORIZED_CORRUPTIONS else None,
)

gc.collect()
torch.cuda.empty_cache()
print(f"Finished. Collect {OUTPUT_ROOT}/APRIL-GAN_artifacts.zip")
